# 05. Objects, dunder methods and design patterns

## Learning goals
1. Customise behaviour with dunder methods.
2. Use dataclasses and abstract base classes.
3. Read the method resolution order.
4. Apply the five SOLID principles and four common patterns.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## Dunder methods
Operators and built-ins call special methods. `a + b` calls `type(a).__add__(a, b)`.
Objects that define `__eq__` must define `__hash__` to stay usable in sets.

In [ ]:
from understanding_python.patterns import Vector
v, w = Vector(3, 4), Vector(1, 1)
assert v + w == Vector(4, 5) and 2 * v == Vector(6, 8)
assert abs(v) == 5.0 and not Vector(0, 0) and bool(v)
assert len({Vector(1, 2), Vector(1, 2)}) == 1
assert repr(v) == "Vector(3, 4)"
try:
    v.z = 1
    raise AssertionError
except AttributeError:
    pass       # __slots__ blocks new attributes

## Dataclasses
`@dataclass` writes `__init__`, `__repr__` and `__eq__`. `frozen=True` adds hashing. Use `default_factory` for mutable defaults.

In [ ]:
from understanding_python.patterns import Point, Cart
import dataclasses
assert Point(1, 2) == Point(1, 2) and Point(1) < Point(2)
assert repr(Point(1, 2)) == "Point(x=1, y=2)"
try:
    Point(1).x = 5
    raise AssertionError
except dataclasses.FrozenInstanceError:
    pass
a, b = Cart(), Cart()
a.add("tea")
assert b.items == [] and [f.name for f in dataclasses.fields(Cart)] == ["items"]

## Abstract base classes and the MRO
An ABC refuses instantiation until every abstract method has an implementation.
Python linearises multiple inheritance with C3. The MRO lists the lookup order.

In [ ]:
from understanding_python.patterns import Shape, Rect, Circle, total_area
try:
    Shape()
    raise AssertionError
except TypeError:
    pass
assert abs(total_area([Rect(2, 3), Circle(1)]) - (6 + 3.141592653589793)) < 1e-12
class A:
    def who(self): return "A"
class B(A):
    def who(self): return "B" + super().who()
class C(A):
    def who(self): return "C" + super().who()
class D(B, C):
    def who(self): return "D" + super().who()
assert [k.__name__ for k in D.__mro__] == ["D", "B", "C", "A", "object"]
assert D().who() == "DBCA"

## SOLID in short
1. **Single responsibility.** One reason to change per class.
2. **Open/closed.** `total_area` accepts new shapes without edits.
3. **Liskov substitution.** Any `Discount` can replace another and the total stays valid.
4. **Interface segregation.** Keep ABCs small.
5. **Dependency inversion.** `Checkout` depends on `Discount`, not on a concrete rule.

In [ ]:
from understanding_python.patterns import Checkout, NoDiscount, Percent, Discount
prices = [20.0, 30.0, 50.0]
assert Checkout(NoDiscount()).total(prices) == 100.0
assert Checkout(Percent(25)).total(prices) == 75.0
class Flat(Discount):
    def apply(self, price): return max(price - 10, 0)
assert Checkout(Flat()).total(prices) == 90.0
for d in (NoDiscount(), Percent(10), Flat()):          # Liskov check
    assert 0 <= d.apply(100.0) <= 100.0

## Patterns: observer, factory, singleton
Observer decouples publishers from subscribers. A factory hides construction. A singleton shares one instance.

In [ ]:
from understanding_python.patterns import EventBus, factory, Singleton
bus, seen = EventBus(), []
bus.subscribe("order", seen.append)
bus.subscribe("order", lambda p: seen.append(("copy", p)))
assert bus.publish("order", 42) == 2 and seen == [42, ("copy", 42)]
assert bus.publish("nobody", 1) == 0
assert isinstance(factory("circle"), Circle) and factory("rect").area() == 6
try:
    factory("hexagon")
    raise AssertionError
except ValueError as e:
    assert "hexagon" in str(e)
assert Singleton() is Singleton()

## Complexity and a benchmark
Method dispatch costs O(1): Python reads the MRO cache. Polymorphic calls add a small constant against inline code.
The check below only confirms that the overhead stays within a modest factor.

In [ ]:
from understanding_python.bench import best_time
shapes = [Rect(2, 3) if i % 2 else Circle(1) for i in range(2000)]
inline = lambda: sum(s.w * s.h if isinstance(s, Rect) else 3.14159 * s.r ** 2 for s in shapes)
poly = lambda: total_area(shapes)
ti, tp = best_time(inline, 20), best_time(poly, 20)
print(f"inline {ti:.5f}s polymorphic {tp:.5f}s")
assert tp < ti * 5